# Sprint 36 — Comparaison PC ↔ board (EWC, Pronostia + Monitoring)

Notebook de synthèse de la comparaison appariée PC ↔ NUCLEO-F439ZI du modèle **EWC**.

**Deux comparaisons appariées à condition fixe** (jamais croisées) :

- **Comparaison A — `all` : board vs PC** (dims natives : Pronostia 13, Monitoring 4).
- **Comparaison B — `5feat` : board vs PC** (sous-ensemble figé historique board).

Chaque comparaison couvre les deux protocoles `frozen` (parité exacte + latence inférence)
et `online` (latence inférence + MAJ CL + parité approchée).

> **Séparation par condition (rework)** : tous les plots comparant PC ↔ board produisent
> désormais **une figure par condition** — un fichier `*_5feat.png` et un `*_all.png` — pour
> comparer PC et board sans mélanger `5feat` et `all` sur les mêmes axes. Les §2 (matrices
> d'accuracy CL) et §4 (courbes d'oubli) incluent maintenant la **board** (matrice partielle
> mesurée : diagonale online + dernière ligne du modèle final frozen).

**Étude secondaire** (§10) : effet du *nombre de features* `5feat` vs `all` — comment les
modèles se comportent sous plus de contraintes (moins de features). C'est la **seule** section
qui compare volontairement les deux conditions sur les mêmes axes.

**Axe INT8 vs FP32 sur board** (§12, rework S3612) : pour les deux conditions, comparaison
de la tête EWC en INT8 (`g_ewc_int8`, flag UART 0x40, chargée depuis les poids FP32 par
S3610) vs FP32 — latence, RAM des poids (Gap 3), métrique préservée, accord INT8↔FP32.

Données : `experiments/exp_S36_summary.json` (S3606) + `exp_S36_PC_*` / `exp_S36_board_*`
(dont `*_int8_*`) / `exp_S36_parity_*`. Figures → `docs/figures/sprint36_pc_board_ewc/`.

In [1]:
import json
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sys
ROOT = Path.cwd()
while not (ROOT / "experiments").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.evaluation.plots import (
    save_figure, plot_accuracy_matrix, plot_forgetting_curve,
    plot_performance_by_task_bar,
)

EXP = ROOT / "experiments"
FIGS = ROOT / "docs/figures/sprint36_pc_board_ewc"
FIGS.mkdir(parents=True, exist_ok=True)

DATASETS = ["pronostia", "monitoring"]
CONDITIONS = ["5feat", "all"]
LATENCY_BUDGET_MS = 100.0          # Gap 2
LATENCY_BUDGET_US = LATENCY_BUDGET_MS * 1000
RAM_BUDGET_BYTES = 65_536          # repère "< 64 Ko" (board réelle = 256 Ko SRAM)

summary = json.load(open(EXP / "exp_S36_summary.json"))["results"]

def load(path):
    p = EXP / path
    return json.load(open(p)) if p.exists() else None

def pc_results(cond, ds):
    return load(f"exp_S36_PC_{cond}_ewc_{ds}/results.json")

def parity(cond, proto, ds):
    return load(f"exp_S36_parity_{cond}_{proto}_{ds}.json")

def g(d, *keys):
    for k in keys:
        if d is None:
            return None
        d = d.get(k)
    return d

# --- Reconstruction de la matrice CL "board" (mesurée, jamais inventée) ----------
# La board n'émet pas une matrice T×T : en ligne elle ne réévalue pas les tâches
# passées. On reconstruit donc une matrice PARTIELLE à partir de mesures réelles :
#   • diagonale  R[i,i] = accuracy online par tâche (au moment où la tâche est apprise) ;
#   • dernière ligne R[T-1,j] = accuracy du modèle FINAL (passe frozen) sur chaque tâche,
#       reconstruite des lignes de parité frozen (pred_board vs true) + bornes de tâche
#       empruntées aux lignes online (idx strictement alignés) ;
#   • autres cases = NaN (non mesurées → grises dans le heatmap).

def board_per_task_online(cond, ds):
    """Accuracy online de la board par tâche = diagonale de la matrice CL."""
    return g(load(f"exp_S36_board_online_{cond}_ewc_{ds}/results.json"), "per_task_board_acc")

def board_frozen_per_task(cond, ds):
    """Accuracy du modèle final (frozen) par tâche = dernière ligne de la matrice CL."""
    fr_rows = g(parity(cond, "frozen", ds), "rows")
    on_rows = g(parity(cond, "online", ds), "rows")
    if not fr_rows or not on_rows or len(fr_rows) != len(on_rows):
        return None
    task_of = {r["idx"]: r["task_id"] for r in on_rows}
    acc = defaultdict(lambda: [0, 0])
    for r in fr_rows:
        t = task_of.get(r["idx"])
        if t is None:
            continue
        acc[t][0] += int(r["pred_board"] == r["true"])
        acc[t][1] += 1
    if not acc:
        return None
    T = max(acc) + 1
    return [acc[t][0] / acc[t][1] if acc[t][1] else np.nan for t in range(T)]

def board_acc_matrix(cond, ds):
    """Matrice CL board partielle (diagonale online + dernière ligne frozen)."""
    diag = board_per_task_online(cond, ds)
    if not diag:
        return None
    T = len(diag)
    M = np.full((T, T), np.nan)
    for i in range(T):
        M[i, i] = diag[i]
    last = board_frozen_per_task(cond, ds)
    if last:
        for j, v in enumerate(last[:T]):
            M[T - 1, j] = v
    return M

def plot_board_forgetting(cond, ds, ax):
    """Courbe d'oubli board : par tâche, points mesurés (online à l'étape j,
    modèle final à l'étape T-1) reliés. Saute les NaN non mesurés."""
    M = board_acc_matrix(cond, ds)
    if M is None:
        ax.text(0.5, 0.5, "board indisponible", ha="center", transform=ax.transAxes)
        ax.set_title(f"Board — {ds} ({cond})")
        return
    T = M.shape[0]
    colors = plt.cm.tab10.colors
    for j in range(T):
        steps = [i for i in range(j, T) if not np.isnan(M[i, j])]
        if not steps:
            continue
        ax.plot(steps, [M[i, j] for i in steps], marker="o",
                color=colors[j % len(colors)], label=f"T{j + 1}")
    ax.set_xticks(range(T)); ax.set_xticklabels([f"After T{i + 1}" for i in range(T)])
    ax.set_xlabel("Training Step"); ax.set_ylabel("Accuracy")
    ax.set_title(f"Board — {ds} ({cond})"); ax.legend(title="Task", fontsize=7)
    ax.grid(True, alpha=0.3)

print("Cellules :", [(ds, c) for ds in DATASETS for c in CONDITIONS])
print("Note : Monitoring 5feat ≡ all (4 features natives, cf. S3601).")
print("Plots séparés par condition (un fichier `*_5feat.png` + un `*_all.png`).")

Cellules : [('pronostia', '5feat'), ('pronostia', 'all'), ('monitoring', '5feat'), ('monitoring', 'all')]
Note : Monitoring 5feat ≡ all (4 features natives, cf. S3601).
Plots séparés par condition (un fichier `*_5feat.png` + un `*_all.png`).


## 1. Accuracy finale par tâche — PC vs board

In [2]:
for cond in CONDITIONS:
    fig, axes = plt.subplots(1, len(DATASETS), figsize=(12, 4.2))
    for ax, ds in zip(np.atleast_1d(axes), DATASETS):
        pc = pc_results(cond, ds)
        online = load(f"exp_S36_board_online_{cond}_ewc_{ds}/results.json")
        pta = g(pc, "per_task_acc") or {}
        task_names = [f"T{int(k)+1}" for k in sorted(pta, key=int)]
        pc_d = {f"T{int(k)+1}": pta[k] for k in sorted(pta, key=int)}
        res = {"PC": pc_d}
        board_pt = g(online, "per_task_board_acc")
        if board_pt:
            res["Board (online)"] = {f"T{i+1}": v for i, v in enumerate(board_pt)}
        plot_performance_by_task_bar(res, task_names, title=f"{ds} ({cond})", ax=ax)
    fig.suptitle(f"Accuracy finale par tâche — PC vs board ({cond})", y=1.03)
    save_figure(fig, FIGS / f"accuracy_per_task_{cond}.png")

[plots] Figure saved → /home/leonard/Documents/ENAC/cl-embedded/docs/figures/sprint36_pc_board_ewc/accuracy_per_task_5feat.png


[plots] Figure saved → /home/leonard/Documents/ENAC/cl-embedded/docs/figures/sprint36_pc_board_ewc/accuracy_per_task_all.png


## 2. Matrices d'accuracy CL — PC vs board

Une figure **par condition** (`5feat`, `all`), lignes = plateforme (PC / board), colonnes =
datasets. La board ne réévalue pas les tâches passées en ligne : sa matrice est donc
**partielle mais entièrement mesurée** — la **diagonale** vient de l'accuracy online par tâche
(au moment où la tâche est apprise) et la **dernière ligne** du modèle final (passe `frozen`)
évalué sur chaque tâche. Les cases non mesurées restent **grises** (aucun chiffre inventé).

In [3]:
for cond in CONDITIONS:
    fig, axes = plt.subplots(2, len(DATASETS), figsize=(11, 9))
    for col, ds in enumerate(DATASETS):
        pc = pc_results(cond, ds)
        m_pc = np.array(g(pc, "acc_matrix"), dtype=float) if g(pc, "acc_matrix") else np.full((3, 3), np.nan)
        plot_accuracy_matrix(m_pc, title=f"PC — {ds} ({cond})", ax=axes[0, col])
        m_b = board_acc_matrix(cond, ds)
        if m_b is None:
            m_b = np.full((3, 3), np.nan)
        plot_accuracy_matrix(m_b, title=f"Board — {ds} ({cond})", ax=axes[1, col])
    fig.suptitle(
        f"Matrices d'accuracy CL — PC vs board ({cond})\n"
        "Board : diagonale = online, dernière ligne = modèle final (frozen), gris = non mesuré",
        y=1.05,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.93])
    save_figure(fig, FIGS / f"cl_accuracy_matrix_{cond}.png")

[plots] Figure saved → /home/leonard/Documents/ENAC/cl-embedded/docs/figures/sprint36_pc_board_ewc/cl_accuracy_matrix_5feat.png


[plots] Figure saved → /home/leonard/Documents/ENAC/cl-embedded/docs/figures/sprint36_pc_board_ewc/cl_accuracy_matrix_all.png


## 3. Accuracy finale vs Oubli (AF) — PC vs board

In [4]:
for cond in CONDITIONS:
    fig, ax = plt.subplots(figsize=(7, 5))
    for ds in DATASETS:
        c = summary[ds][cond]
        af_pc, acc_pc = g(c, "pc", "af"), g(c, "pc", "acc_final")
        if af_pc is not None and acc_pc is not None:
            ax.scatter(af_pc, acc_pc, marker="o", s=90, color="tab:blue")
            ax.annotate(ds[:4], (af_pc, acc_pc), fontsize=8, xytext=(4, 4),
                        textcoords="offset points")
        af_b, acc_b = g(c, "board_online", "online_forgetting"), g(c, "board_online", "online_accuracy")
        if af_b is not None and acc_b is not None:
            ax.scatter(af_b, acc_b, marker="^", s=90, color="tab:orange")
            ax.annotate(ds[:4], (af_b, acc_b), fontsize=8, xytext=(4, -10),
                        textcoords="offset points")
    ax.scatter([], [], marker="o", color="tab:blue", label="PC")
    ax.scatter([], [], marker="^", color="tab:orange", label="Board (online)")
    ax.set_xlabel("Average Forgetting (AF)"); ax.set_ylabel("Accuracy finale")
    ax.set_title(f"Acc finale vs Oubli — PC vs board ({cond})"); ax.legend(); ax.grid(alpha=0.3)
    save_figure(fig, FIGS / f"accfinal_vs_forgetting_{cond}.png")

[plots] Figure saved → /home/leonard/Documents/ENAC/cl-embedded/docs/figures/sprint36_pc_board_ewc/accfinal_vs_forgetting_5feat.png


[plots] Figure saved → /home/leonard/Documents/ENAC/cl-embedded/docs/figures/sprint36_pc_board_ewc/accfinal_vs_forgetting_all.png


## 4. Courbes d'oubli par tâche — PC vs board

Une figure **par condition**, lignes = plateforme (PC / board), colonnes = datasets. Côté
board, chaque tâche relie ses deux points mesurés : accuracy **online** (étape où la tâche est
apprise) → accuracy du **modèle final** (passe `frozen`). Une courbe plate = pas d'oubli.

In [5]:
for cond in CONDITIONS:
    fig, axes = plt.subplots(2, len(DATASETS), figsize=(12, 8))
    for col, ds in enumerate(DATASETS):
        pc = pc_results(cond, ds)
        m = np.array(g(pc, "acc_matrix"), dtype=float) if g(pc, "acc_matrix") else np.full((3, 3), np.nan)
        plot_forgetting_curve(m, title=f"PC — {ds} ({cond})", ax=axes[0, col])
        plot_board_forgetting(cond, ds, axes[1, col])
    fig.suptitle(f"Courbes d'oubli par tâche — PC vs board ({cond})", y=1.02)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    save_figure(fig, FIGS / f"forgetting_curves_{cond}.png")

[plots] Figure saved → /home/leonard/Documents/ENAC/cl-embedded/docs/figures/sprint36_pc_board_ewc/forgetting_curves_5feat.png


[plots] Figure saved → /home/leonard/Documents/ENAC/cl-embedded/docs/figures/sprint36_pc_board_ewc/forgetting_curves_all.png


## 5. Latence board : inférence (frozen) vs inférence+MAJ (online)

In [6]:
for cond in CONDITIONS:
    labels, lat_inf, lat_upd = [], [], []
    for ds in DATASETS:
        c = summary[ds][cond]
        labels.append(ds)
        lat_inf.append(g(c, "board_frozen", "latency_us_p50"))
        lat_upd.append(g(c, "board_online", "latency_us_p50"))
    x = np.arange(len(labels)); w = 0.38
    fig, ax = plt.subplots(figsize=(7, 4.8))
    ax.bar(x - w/2, [v or 0 for v in lat_inf], w, label="Inférence seule (frozen)", color="tab:green")
    ax.bar(x + w/2, [v or 0 for v in lat_upd], w, label="Inférence + MAJ CL (online)", color="tab:red")
    ax.axhline(LATENCY_BUDGET_US, color="k", ls="--", label="Gap 2 (100 ms)")
    ax.set_yscale("log"); ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_ylabel("Latence P50 (µs, log)")
    ax.set_title(f"Latence inférence vs inférence+MAJ — board ({cond})")
    ax.legend(); ax.grid(alpha=0.3, axis="y")
    save_figure(fig, FIGS / f"latency_inference_vs_update_{cond}.png")

[plots] Figure saved → /home/leonard/Documents/ENAC/cl-embedded/docs/figures/sprint36_pc_board_ewc/latency_inference_vs_update_5feat.png
[plots] Figure saved → /home/leonard/Documents/ENAC/cl-embedded/docs/figures/sprint36_pc_board_ewc/latency_inference_vs_update_all.png


## 6. Latence PC vs board (inférence)

In [7]:
for cond in CONDITIONS:
    labels, lat_pc, lat_board = [], [], []
    for ds in DATASETS:
        c = summary[ds][cond]
        labels.append(ds)
        ms = g(c, "pc", "inference_latency_ms")
        lat_pc.append(ms * 1000 if isinstance(ms, (int, float)) else None)
        lat_board.append(g(c, "board_frozen", "latency_us_p50"))
    x = np.arange(len(labels)); w = 0.38
    fig, ax = plt.subplots(figsize=(7, 4.8))
    ax.bar(x - w/2, [v or 0 for v in lat_pc], w, label="PC (inférence)", color="tab:blue")
    ax.bar(x + w/2, [v or 0 for v in lat_board], w, label="Board (inférence)", color="tab:green")
    ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylabel("Latence (µs)")
    ax.set_title(f"Latence inférence PC vs board ({cond})"); ax.legend(); ax.grid(alpha=0.3, axis="y")
    if not any(lat_pc):
        ax.text(0.5, 0.9, "Latence PC non renseignée pour la plupart des cellules",
                transform=ax.transAxes, ha="center", color="gray")
    save_figure(fig, FIGS / f"latency_pc_vs_board_{cond}.png")

[plots] Figure saved → /home/leonard/Documents/ENAC/cl-embedded/docs/figures/sprint36_pc_board_ewc/latency_pc_vs_board_5feat.png
[plots] Figure saved → /home/leonard/Documents/ENAC/cl-embedded/docs/figures/sprint36_pc_board_ewc/latency_pc_vs_board_all.png


## 7. Accuracy vs RAM — PC (`ram_peak_bytes`) vs board (`.bss`)

Zone grisée = repère « < 64 Ko ». La board réelle dispose de 256 Ko SRAM ; le `.bss` EWC
(100–145 Ko) reste ≪ 256 Ko.

In [8]:
for cond in CONDITIONS:
    fig, ax = plt.subplots(figsize=(8, 5))
    for ds in DATASETS:
        c = summary[ds][cond]
        ram_pc, acc_pc = g(c, "pc", "ram_peak_bytes"), g(c, "pc", "acc_final")
        if ram_pc and acc_pc is not None:
            ax.scatter(ram_pc, acc_pc, marker="o", s=90, color="tab:blue")
            ax.annotate(ds[:4], (ram_pc, acc_pc), fontsize=8,
                        xytext=(4, 4), textcoords="offset points")
        bss, acc_b = g(c, "board_frozen", "bss_bytes"), g(c, "board_frozen", "online_accuracy")
        if bss and acc_b is not None:
            ax.scatter(bss, acc_b, marker="^", s=90, color="tab:orange")
            ax.annotate(ds[:4], (bss, acc_b), fontsize=8,
                        xytext=(4, -10), textcoords="offset points")
    ax.axvspan(0, RAM_BUDGET_BYTES, color="green", alpha=0.08, label="< 64 Ko")
    ax.scatter([], [], marker="o", color="tab:blue", label="PC (ram_peak)")
    ax.scatter([], [], marker="^", color="tab:orange", label="Board (.bss)")
    ax.set_xlabel("RAM (octets)"); ax.set_ylabel("Accuracy")
    ax.set_title(f"Accuracy vs RAM — PC vs board ({cond})"); ax.legend(); ax.grid(alpha=0.3)
    save_figure(fig, FIGS / f"accuracy_vs_ram_{cond}.png")

[plots] Figure saved → /home/leonard/Documents/ENAC/cl-embedded/docs/figures/sprint36_pc_board_ewc/accuracy_vs_ram_5feat.png


[plots] Figure saved → /home/leonard/Documents/ENAC/cl-embedded/docs/figures/sprint36_pc_board_ewc/accuracy_vs_ram_all.png


## 8. F1 et ROC-AUC — PC vs board (frozen)

In [9]:
for cond in CONDITIONS:
    labels, f1_pc, f1_b, roc_pc = [], [], [], []
    for ds in DATASETS:
        c = summary[ds][cond]
        labels.append(ds)
        f1_pc.append(g(c, "pc", "f1_faulty"))
        f1_b.append(g(c, "board_frozen", "f1_faulty"))
        roc_pc.append(g(c, "pc", "roc_auc"))
    x = np.arange(len(labels)); w = 0.27
    fig, ax = plt.subplots(figsize=(8, 4.8))
    ax.bar(x - w, [v or 0 for v in f1_pc], w, label="F1 PC", color="tab:blue")
    ax.bar(x,     [v or 0 for v in f1_b], w, label="F1 board", color="tab:orange")
    ax.bar(x + w, [v or 0 for v in roc_pc], w, label="ROC-AUC PC", color="tab:green")
    ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score"); ax.set_title(f"F1 (faulty) et ROC-AUC — PC vs board ({cond})")
    ax.legend(); ax.grid(alpha=0.3, axis="y")
    save_figure(fig, FIGS / f"f1_rocauc_pc_vs_board_{cond}.png")

[plots] Figure saved → /home/leonard/Documents/ENAC/cl-embedded/docs/figures/sprint36_pc_board_ewc/f1_rocauc_pc_vs_board_5feat.png


[plots] Figure saved → /home/leonard/Documents/ENAC/cl-embedded/docs/figures/sprint36_pc_board_ewc/f1_rocauc_pc_vs_board_all.png


## 9. Parité des prédictions PC ↔ board

Une figure **par condition**. À gauche : taux de concordance par dataset (frozen = exact
1.000 ; online = approché). À droite : matrice de confusion des **désaccords online**
(pred_pc vs pred_board) pour `pronostia`.

In [10]:
for cond in CONDITIONS:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.6))
    labels, fr, on = [], [], []
    for ds in DATASETS:
        labels.append(ds)
        fr.append(g(parity(cond, "frozen", ds), "parity_rate"))
        on.append(g(parity(cond, "online", ds), "parity_rate"))
    x = np.arange(len(labels)); w = 0.38
    ax1.bar(x - w/2, [v if v is not None else 0 for v in fr], w, label="frozen", color="tab:green")
    ax1.bar(x + w/2, [v if v is not None else 0 for v in on], w, label="online", color="tab:red")
    ax1.set_xticks(x); ax1.set_xticklabels(labels); ax1.set_ylim(0.9, 1.005)
    ax1.set_ylabel("parity_rate"); ax1.set_title(f"Concordance PC↔board ({cond})")
    ax1.legend(); ax1.grid(alpha=0.3, axis="y")

    par = parity(cond, "online", "pronostia")
    if par and par.get("rows"):
        conf = np.zeros((2, 2), dtype=int)
        for r in par["rows"]:
            conf[int(r["pred_pc"]), int(r["pred_board"])] += 1
        im = ax2.imshow(conf, cmap="Blues")
        for i in range(2):
            for j in range(2):
                ax2.text(j, i, conf[i, j], ha="center", va="center",
                         color="white" if conf[i, j] > conf.max()/2 else "black")
        ax2.set_xticks([0, 1]); ax2.set_yticks([0, 1])
        ax2.set_xlabel("pred_board"); ax2.set_ylabel("pred_pc")
        ax2.set_title(f"Confusion online pronostia ({cond}, mismatch={par['mismatch_count']})")
        fig.colorbar(im, ax=ax2, fraction=0.046)
    else:
        ax2.text(0.5, 0.5, "parité online indisponible", ha="center", transform=ax2.transAxes)
    save_figure(fig, FIGS / f"prediction_parity_{cond}.png")

[plots] Figure saved → /home/leonard/Documents/ENAC/cl-embedded/docs/figures/sprint36_pc_board_ewc/prediction_parity_5feat.png
[plots] Figure saved → /home/leonard/Documents/ENAC/cl-embedded/docs/figures/sprint36_pc_board_ewc/prediction_parity_all.png


## 10. Étude secondaire — effet du nombre de features (`5feat` vs `all`)

Comparaison *transverse* aux conditions (≠ comparaisons appariées A/B ci-dessus) : comment
le modèle se comporte sous plus de contraintes (moins de features). Pronostia distingue les
deux conditions (5 vs 13 features) ; Monitoring est identique (4 features natives,
`5feat ≡ all`).


In [11]:
ds = "pronostia"
metrics = {
    "acc_final PC": [g(summary[ds][c], "pc", "acc_final") for c in CONDITIONS],
    "parity online": [g(parity(c, "online", ds), "parity_rate") for c in CONDITIONS],
    "lat online (µs)": [g(summary[ds][c], "board_online", "latency_us_p50") for c in CONDITIONS],
}
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (name, vals) in zip(axes, metrics.items()):
    ax.bar(CONDITIONS, [v if v is not None else 0 for v in vals],
           color=["tab:blue", "tab:orange"])
    ax.set_title(name); ax.grid(alpha=0.3, axis="y")
    for i, v in enumerate(vals):
        if v is not None:
            ax.text(i, v, f"{v:.3f}" if v < 100 else f"{v:.0f}", ha="center", va="bottom", fontsize=8)
fig.suptitle(f"Effet condition 5feat vs all — {ds}", y=1.04)
save_figure(fig, FIGS / "condition_5feat_vs_all.png")

[plots] Figure saved → /home/leonard/Documents/ENAC/cl-embedded/docs/figures/sprint36_pc_board_ewc/condition_5feat_vs_all.png


## 11. Tableau récapitulatif — tous les métriques

In [12]:
recap = []
for ds in DATASETS:
    for cond in CONDITIONS:
        c = summary[ds][cond]
        recap.append({
            "dataset": ds, "condition": cond,
            "PC acc_final": g(c, "pc", "acc_final"),
            "PC AF": g(c, "pc", "af"),
            "PC F1": g(c, "pc", "f1_faulty"),
            "PC ROC-AUC": g(c, "pc", "roc_auc"),
            "PC RAM (B)": g(c, "pc", "ram_peak_bytes"),
            "board acc": g(c, "board_frozen", "online_accuracy"),
            "board F1": g(c, "board_frozen", "f1_faulty"),
            ".bss (B)": g(c, "board_frozen", "bss_bytes"),
            "lat inf (µs)": g(c, "board_frozen", "latency_us_p50"),
            "lat inf+MAJ (µs)": g(c, "board_online", "latency_us_p50"),
            "parity frozen": g(parity(cond, "frozen", ds), "parity_rate"),
            "parity online": g(parity(cond, "online", ds), "parity_rate"),
            "Δacc PC↔board": g(c, "delta_pc_board", "acc_final"),
        })
df = pd.DataFrame(recap)
pd.set_option("display.float_format", lambda v: f"{v:.4f}" if isinstance(v, float) else str(v))
df

,dataset,condition,PC acc_final,PC AF,PC F1,PC ROC-AUC,PC RAM (B),board acc,board F1,.bss (B),lat inf (µs),lat inf+MAJ (µs),parity frozen,parity online,Δacc PC↔board
0,pronostia,5feat,0.9887,0.0100,0.9164,0.9955,59734496,0.9821,0.9164,105036,50.0000,251.0000,1.0000,0.9754,0.0066
1,pronostia,all,0.9834,0.0050,0.9180,0.9974,133463,0.9831,0.9180,144516,65.0000,340.0000,1.0000,0.9626,0.0003
2,monitoring,5feat,0.9791,0.0000,0.9194,0.9879,128363,0.9846,0.9194,100152,48.0000,239.0000,1.0000,0.9887,0.0055
3,monitoring,all,0.9791,0.0000,0.9194,0.9879,222375,0.9846,0.9194,100152,48.0000,239.0000,1.0000,0.9887,0.0055


## 12. INT8 vs FP32 sur board — Gap 3

Comparaison de la tête EWC **INT8** (`g_ewc_int8`, flag 0x40) vs **FP32** sur la NUCLEO, pour
les deux datasets, en `frozen` et `online`, **une figure par condition** (`5feat`, `all`).
Métriques (modèle Sprint 28/29) : latence DWT, RAM des tenseurs de poids (FP32/INT8,
ratio ≈ 4×), métrique préservée (F1) et **accord INT8↔FP32 board**. Tant que la NUCLEO n'a pas
streamé l'INT8, ces champs sont `null` (règle « aucun chiffre inventé ») et les plots affichent
un repère « à mesurer ».

In [13]:
# Collecte des cellules INT8 (frozen + online) présentes dans le summary, par condition.
def int8_rows_for(cond):
    rows = []
    for ds in DATASETS:
        fr = g(summary, ds, cond, "board_frozen_int8") or {}
        on = g(summary, ds, cond, "board_online_int8") or {}
        fr_fp = g(summary, ds, cond, "board_frozen") or {}
        on_fp = g(summary, ds, cond, "board_online") or {}
        rows.append({
            "cell": ds, "ds": ds, "cond": cond,
            "lat_frozen_int8": fr.get("latency_us_p50"),
            "lat_frozen_fp32": fr_fp.get("latency_us_p50"),
            "lat_online_int8": on.get("latency_us_p50"),
            "lat_online_fp32": on_fp.get("latency_us_p50"),
            "f1_int8": fr.get("f1_faulty"), "f1_fp32": fr_fp.get("f1_faulty"),
            "acc_int8": fr.get("online_accuracy"), "acc_fp32": fr_fp.get("online_accuracy"),
            "agree": fr.get("agreement_int8_vs_fp32"),
            "ram_fp32": fr.get("ram_weights_fp32_bytes"),
            "ram_int8": fr.get("ram_weights_int8_bytes"),
            "ram_ratio": fr.get("ram_ratio_fp32_over_int8"),
            "gap3_ram_ok": fr.get("gap3_ram_ok"),
        })
    return rows

all_int8 = {c: int8_rows_for(c) for c in CONDITIONS}
HAS_INT8 = any(r["lat_frozen_int8"] is not None or r["f1_int8"] is not None
               for rows in all_int8.values() for r in rows)
pd.DataFrame([r for rows in all_int8.values() for r in rows]).set_index(["cond", "cell"])

ds  lat_frozen_int8  lat_frozen_fp32  \
cond  cell                                                       
5feat pronostia    pronostia          53.0000          50.0000   
      monitoring  monitoring          51.0000          48.0000   
all   pronostia    pronostia          68.0000          65.0000   
      monitoring  monitoring          51.0000          48.0000   

                  lat_online_int8  lat_online_fp32  f1_int8  f1_fp32  \
cond  cell                                                             
5feat pronostia          462.0000         251.0000   0.1380   0.9164   
      monitoring         440.0000         239.0000   0.1337   0.9194   
all   pronostia          639.0000         340.0000   0.1502   0.9180   
      monitoring         440.0000         239.0000   0.1337   0.9194   

                  acc_int8  acc_fp32  agree  ram_fp32  ram_int8  ram_ratio  \
cond  cell                                                                   
5feat pronostia     0.7463    0.9821 0.7364      2816       704     4.0000   
      monitoring    0.5913    0.9846 0.5955      2688       672     4.0000   
all   pronostia     0.6830    0.9831 0.6813      3840       960     4.0000   
      monitoring    0.5913    0.9846 0.5955      2688       672     4.0000   

                  gap3_ram_ok  
cond  cell                     
5feat pronostia          True  
      monitoring         True  
all   pronostia          True  
      monitoring         True

In [14]:
# Plots INT8 vs FP32 : une figure PAR CONDITION (latence frozen+online, RAM poids, F1+accord).
for cond in CONDITIONS:
    int8_rows = all_int8[cond]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))

    if not HAS_INT8:
        for ax in axes:
            ax.text(0.5, 0.5, "INT8 non encore mesuré sur board\n(NUCLEO non branchée — à mesurer)",
                    ha="center", va="center", fontsize=11, color="grey")
            ax.set_xticks([]); ax.set_yticks([])
    else:
        cells = [r["cell"] for r in int8_rows]
        x = np.arange(len(cells)); w = 0.2

        def _v(key):
            return [r[key] if isinstance(r[key], (int, float)) else np.nan for r in int8_rows]

        # (a) Latence INT8 vs FP32, frozen + online, échelle log + ligne Gap 2.
        ax = axes[0]
        ax.bar(x - 1.5*w, _v("lat_frozen_fp32"), w, label="frozen FP32")
        ax.bar(x - 0.5*w, _v("lat_frozen_int8"), w, label="frozen INT8")
        ax.bar(x + 0.5*w, _v("lat_online_fp32"), w, label="online FP32")
        ax.bar(x + 1.5*w, _v("lat_online_int8"), w, label="online INT8")
        ax.axhline(LATENCY_BUDGET_US, color="red", ls="--", lw=1, label="Gap 2 (100 ms)")
        ax.set_yscale("log"); ax.set_ylabel("latence P50 (µs, log)")
        ax.set_xticks(x); ax.set_xticklabels(cells, rotation=30, ha="right")
        ax.set_title("(a) Latence INT8 vs FP32"); ax.legend(fontsize=7)

        # (b) RAM des poids FP32 vs INT8 (+ ratio annoté).
        ax = axes[1]
        ax.bar(x - w/2, _v("ram_fp32"), w, label="poids FP32 (B)")
        ax.bar(x + w/2, _v("ram_int8"), w, label="poids INT8 (B)")
        for i, r in enumerate(int8_rows):
            if isinstance(r["ram_ratio"], (int, float)):
                ax.text(i, max(r["ram_fp32"] or 0, 1), f"×{r['ram_ratio']:.1f}",
                        ha="center", va="bottom", fontsize=8)
        ax.set_ylabel("octets"); ax.set_xticks(x)
        ax.set_xticklabels(cells, rotation=30, ha="right")
        ax.set_title("(b) RAM poids — Gap 3"); ax.legend(fontsize=8)

        # (c) F1 préservée INT8 vs FP32 + accord INT8↔FP32.
        ax = axes[2]
        ax.bar(x - w/2, _v("f1_fp32"), w, label="F1 FP32")
        ax.bar(x + w/2, _v("f1_int8"), w, label="F1 INT8")
        ax2 = ax.twinx()
        ax2.plot(x, _v("agree"), "k^--", label="accord INT8↔FP32")
        ax2.set_ylim(0, 1.05); ax2.set_ylabel("accord")
        ax.set_ylabel("F1_faulty"); ax.set_xticks(x)
        ax.set_xticklabels(cells, rotation=30, ha="right")
        ax.set_title("(c) Métrique préservée + accord"); ax.legend(fontsize=8, loc="lower left")

    fig.suptitle(f"INT8 vs FP32 sur board ({cond})", y=1.04)
    fig.tight_layout()
    save_figure(fig, FIGS / f"int8_vs_fp32_board_{cond}.png")
    plt.show()

[plots] Figure saved → /home/leonard/Documents/ENAC/cl-embedded/docs/figures/sprint36_pc_board_ewc/int8_vs_fp32_board_5feat.png


[plots] Figure saved → /home/leonard/Documents/ENAC/cl-embedded/docs/figures/sprint36_pc_board_ewc/int8_vs_fp32_board_all.png


## 13. RAM du modèle EWC sur board — inférence vs inférence + MAJ

Empreinte RAM des **tenseurs du modèle** alloués statiquement dans `EWCHead`
([firmware `inc/ewc_head.h`](../../../firmware/stm32f4_blink/inc/ewc_head.h), FP32, 4 B) :

- **Inférence** = poids + biais (`w1,b1,w2,b2,w3,b3`) — seuls nécessaires au `ewc_forward`.
- **Inférence + MAJ CL** = ajoute l'état de consolidation EWC : diagonale de **Fisher**
  (`fisher1/2/3`) + **ancre θ\*** (`star_w1/2/3`), requis par `ewc_sgd_step` /
  `ewc_consolidate`.

Calculé à partir des `#define` firmware (`EWC_H1=32`, `EWC_H2=16`, `EWC_OUT=2`) et de la
dimension d'entrée réelle par cellule (`EWC_IN` = nombre de features) — **pas un chiffre
inventé**, c'est la taille exacte des buffers compilés. Une figure par condition.

In [15]:
# Tailles depuis le firmware inc/ewc_head.h (struct EWCHead).
EWC_H1, EWC_H2, EWC_OUT = 32, 16, 2
FP32_B = 4

def ewc_model_ram(n_in):
    """RAM (octets) des tenseurs EWCHead : (inférence, inférence+MAJ).
    inférence = poids + biais ; +MAJ ajoute Fisher (poids) + ancre θ* (poids)."""
    weights_bias = (EWC_H1 * n_in + EWC_H1) + (EWC_H2 * EWC_H1 + EWC_H2) + (EWC_OUT * EWC_H2 + EWC_OUT)
    fisher = EWC_H1 * n_in + EWC_H2 * EWC_H1 + EWC_OUT * EWC_H2  # diag Fisher (poids only)
    star = fisher                                               # ancre θ* (poids only)
    return FP32_B * weights_bias, FP32_B * (weights_bias + fisher + star)

def board_nfeat(cond, ds):
    return g(load(f"exp_S36_board_frozen_{cond}_ewc_{ds}/results.json"), "n_features")

for cond in CONDITIONS:
    labels, ram_inf, ram_upd = [], [], []
    for ds in DATASETS:
        n_in = board_nfeat(cond, ds)
        if n_in is None:
            continue
        ri, ru = ewc_model_ram(n_in)
        labels.append(f"{ds}\n({n_in} feat.)"); ram_inf.append(ri); ram_upd.append(ru)
    x = np.arange(len(labels)); w = 0.38
    fig, ax = plt.subplots(figsize=(7, 4.8))
    b1 = ax.bar(x - w/2, ram_inf, w, label="Inférence (poids + biais)", color="tab:green")
    b2 = ax.bar(x + w/2, ram_upd, w, label="Inférence + MAJ (+ Fisher + θ*)", color="tab:red")
    for bars in (b1, b2):
        for r in bars:
            ax.text(r.get_x() + r.get_width()/2, r.get_height(), f"{int(r.get_height())} B",
                    ha="center", va="bottom", fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_ylabel("RAM tenseurs modèle (octets, FP32)")
    ax.set_title(f"RAM modèle EWC board — inférence vs inférence+MAJ ({cond})")
    ax.legend(); ax.grid(alpha=0.3, axis="y")
    ax.margins(y=0.18)
    save_figure(fig, FIGS / f"board_model_ram_inf_vs_update_{cond}.png")

[plots] Figure saved → /home/leonard/Documents/ENAC/cl-embedded/docs/figures/sprint36_pc_board_ewc/board_model_ram_inf_vs_update_5feat.png
[plots] Figure saved → /home/leonard/Documents/ENAC/cl-embedded/docs/figures/sprint36_pc_board_ewc/board_model_ram_inf_vs_update_all.png


## 14. INT8 vs FP32 sur board — Accuracy & F1

Comparaison directe de la **métrique préservée** entre la tête EWC INT8 (`g_ewc_int8`) et FP32
sur la board (passe `frozen`), une figure par condition : à gauche **accuracy** INT8 vs FP32,
à droite **F1 (faulty)** INT8 vs FP32. Rappel S3610 : la PTQ embarquée de la tête binaire
**dégrade fortement** la métrique (F1 ≈ 0.1–0.15 vs ≈ 0.92 FP32), distincte du fake-quant QAT
PC (Sprint 28, Δ≤0.006 préservé).

In [16]:
for cond in CONDITIONS:
    rows = all_int8[cond]
    cells = [r["cell"] for r in rows]
    x = np.arange(len(cells)); w = 0.38
    fig, (axa, axf) = plt.subplots(1, 2, figsize=(12, 4.6))

    def _v(key):
        return [r[key] if isinstance(r[key], (int, float)) else np.nan for r in rows]

    for ax, (fp_key, i8_key, title) in zip(
        (axa, axf),
        [("acc_fp32", "acc_int8", "Accuracy"), ("f1_fp32", "f1_int8", "F1 (faulty)")],
    ):
        ax.bar(x - w/2, _v(fp_key), w, label=f"{title} FP32", color="tab:blue")
        ax.bar(x + w/2, _v(i8_key), w, label=f"{title} INT8", color="tab:orange")
        for i, (vf, vi) in enumerate(zip(_v(fp_key), _v(i8_key))):
            if not np.isnan(vf):
                ax.text(i - w/2, vf, f"{vf:.2f}", ha="center", va="bottom", fontsize=8)
            if not np.isnan(vi):
                ax.text(i + w/2, vi, f"{vi:.2f}", ha="center", va="bottom", fontsize=8)
        ax.set_xticks(x); ax.set_xticklabels(cells); ax.set_ylim(0, 1.05)
        ax.set_ylabel(title); ax.set_title(f"{title} — INT8 vs FP32")
        ax.legend(); ax.grid(alpha=0.3, axis="y")

    fig.suptitle(f"INT8 vs FP32 sur board — accuracy & F1 ({cond})", y=1.02)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    save_figure(fig, FIGS / f"int8_vs_fp32_acc_f1_{cond}.png")

[plots] Figure saved → /home/leonard/Documents/ENAC/cl-embedded/docs/figures/sprint36_pc_board_ewc/int8_vs_fp32_acc_f1_5feat.png


[plots] Figure saved → /home/leonard/Documents/ENAC/cl-embedded/docs/figures/sprint36_pc_board_ewc/int8_vs_fp32_acc_f1_all.png


---
**Synthèse** : parité frozen **exacte (1.000)** sur les 8 cellules ; parité online **approchée**
(0.96–0.99, divergence float32 board / float64 PC) ; toutes les latences (inférence 48–65 µs ;
inférence+MAJ 239–340 µs) **≪ 100 ms** ⇒ **Gap 2 préservé**. Δacc_final PC↔board ≤ 0.007.